# Reconstruction conditioning for normalized overlap-add

This notebook slows down one narrow point from the repo's new reconstruction-conditioning sidecar: exact same-window overlap-add reconstruction is not the same thing as a numerically calm reconstruction path.


## The setup

For a window `w[n]` and hop `H`, same-window analysis/synthesis with pointwise normalization uses the denominator

`d[n] = Σ_m w^2[n - mH]`.

If `d[n] > 0`, the reconstruction is exact in ideal arithmetic. But if frame coefficients carry small additive error before synthesis windowing, the local output noise standard deviation scales like `1 / sqrt(d[n])`.


In [ ]:
from pprint import pprint

from windowlab.reconstruct import (
    build_reference_signal,
    periodic_same_window_reconstruction,
    reconstruction_condition_summary,
    simulated_relative_noise_gain,
)
from windowlab.windows import build_window


In [ ]:
for hop in (64, 32, 16):
    print(f'\nHop {hop} (overlap {(1 - hop/128)*100:.1f}%)')
    for name in ('hann', 'hamming', 'blackman', 'blackman-harris', 'nuttall', 'flattop'):
        summary = reconstruction_condition_summary(build_window(name, 128), hop)
        print(
            f'{name:16s} floor={summary.min_denominator_fraction:0.6f} '
            f'rms_gain={summary.rms_relative_noise_gain:0.6f} '
            f'worst_gain={summary.worst_relative_noise_gain:0.6f}'
        )


## Exactness check

The algebraic reconstruction identity really does hold in the periodic test path used by the repo. The error stays down near machine precision.


In [ ]:
signal = build_reference_signal(1024)
run = periodic_same_window_reconstruction(signal, build_window('blackman-harris', 128), 32)
{'rmse': run.rmse, 'max_abs_error': run.max_abs_error}


## Monte Carlo sanity check

The repo also includes a simple noise simulation. It is not the main metric, but it is a useful guardrail: the simulated relative gain should sit close to the analytic RMS prediction.


In [ ]:
analytic = reconstruction_condition_summary(build_window('flattop', 128), 32).rms_relative_noise_gain
simulated = simulated_relative_noise_gain(build_window('flattop', 128), 32, periods=64, coefficient_noise_std=1e-6, seed=7)
{'analytic_rms_gain': analytic, 'simulated_rms_gain': simulated}


## Readout

The practical conclusion is short:

- half-overlap is rough for the heavier windows
- quarter-hop already makes Hann, Hamming, Blackman, Blackman-Harris, and Nuttall calm
- flat-top is still the holdout at quarter-hop
- one-eighth hop makes the whole table essentially flat again
